# Первые эксперименты: пробы промптов
**Автор:** Алёна Кизименко  
**Дата:** 2026-09-15  
**Модель:** GPT-4o-mini

In [ ]:
import openai
import json
import time
from datetime import datetime

# Настройка клиента (замените на ваш реальный ключ)
client = openai.OpenAI(api_key="your-api-key")
current_date = datetime.now().strftime("%Y-%m-%d")
print(f"Текущая дата для промптов: {current_date}")

In [ ]:
examples = [
    {
        "id": 1,
        "pain": "Нестандартная дата и цена с условием",
        "text": "Ребята, в субботу в 19 разбираем тревогу у подростков. Пространство «Точка», Пушкина 12. Стоимость 2500₽ (для своих 2000). Записывайтесь в личку."
    },
    {
        "id": 2,
        "pain": "Отсутствие чёткой структуры",
        "text": "Всем привет! В эту среду встречаемся, поговорим про выгорание. Место уточню позже, ориентировочно центр. Цена символическая — 500₽. Приходите, будет полезно!"
    },
    {
        "id": 3,
        "pain": "Длинный текст с лишней информацией",
        "text": "Дорогие друзья, рад сообщить, что в рамках нашего цикла встреч по психологии отношений мы проводим очередную встречу. Тема: «Как строить здоровые границы в семье». Дата: 25 ноября, вторник, начало в 18:30. Место: коворкинг «Среда», ул. Ленина 45, 3 этаж. Стоимость участия: 3000₽. Для постоянных участников — 2500₽. Запись через личные сообщения или по телефону +7 (999) 123-45-67. Количество мест ограничено — всего 10 человек. Не забудьте взять с собой блокнот и хорошее настроение!"
    }
]

## Подход A: Few-shot prompting

In [ ]:
few_shot_prompt = """
Ты — парсер анонсов мероприятий. Извлеки из текста следующие поля в JSON:
- date_start: YYYY-MM-DD
- time_start: HH:MM
- venue: адрес места
- price: число в рублях
- price_discount: число или null
- title: заголовок встречи (до 100 символов)

Пример 1:
Текст: "Встреча в субботу 15 ноября в 19:00, пространство «Точка», Пушкина 12, 2500₽."
JSON: {"date_start": "2025-11-15", "time_start": "19:00", "venue": "Пушкина 12", "price": 2500, "price_discount": null, "title": "Встреча"}

Пример 2:
Текст: "В среду в 18:00, коворкинг «Среда», Ленина 45, 3000₽ (для своих 2500)."
JSON: {"date_start": "2025-11-19", "time_start": "18:00", "venue": "Ленина 45", "price": 3000, "price_discount": 2500, "title": "Встреча"}

Текст анонса:
"""
few_shot_prompt += '"""\n{body}\n"""\n\nВерни только JSON, без пояснений.'

print("=== Результаты Few-shot prompting ===")
for ex in examples:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": few_shot_prompt.format(body=ex["text"])}],
        temperature=0.1,
        max_tokens=500
    )
    print(f"\nПример {ex['id']}:")
    print(response.choices[0].message.content)
    print("-" * 50)

In [ ]:
# Пример 4: Правка после публикации
example_4 = {
    "id": 4,
    "pain": "Рассинхронизация при правках",
    "original_text": "Встреча в субботу 15 ноября в 19:00, пространство «Точка», Пушкина 12, 2500₽.",
    "edit_text": "Внимание! Переносим встречу на воскресенье 16 ноября, 15:00. Место то же."
}

# Сценарий: эксперт правит только время и дату, остальное не повторяет
print("=== Тест правки (Пример 4) ===\n")

# Stage 1: Извлекаем данные из правки
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example_4["edit_text"]}
    ],
    functions=functions,
    function_call={"name": "extract_event_data"},
    temperature=0.1
)

edited_data = json.loads(response.choices[0].message.function_call.arguments)
print("Извлечённые данные из правки:")
print(json.dumps(edited_data, ensure_ascii=False, indent=2))

# Ключевой вопрос: сохранился ли price_discount и venue из исходного анонса?
print("\n⚠️ Проблема: модель не видит цену и полное название места в правке!")
print("Решение: нужно мержить новый JSON с исходным (хранить в БД)")

## Подход B: Function Calling

In [ ]:
functions = [
    {
        "name": "extract_event_data",
        "description": "Извлечь структурированные данные из анонса мероприятия",
        "parameters": {
            "type": "object",
            "properties": {
                "date_start": {"type": "string", "description": "Дата начала в формате YYYY-MM-DD"},
                "date_end": {"type": "string", "description": "Дата окончания или null"},
                "time_start": {"type": "string", "description": "Время начала в формате HH:MM"},
                "venue": {"type": "string", "description": "Адрес места проведения"},
                "price": {"type": "number", "description": "Стоимость участия в рублях"},
                "price_discount": {"type": "number", "description": "Льготная цена или null"},
                "title": {"type": "string", "description": "Заголовок встречи, до 100 символов"}
            },
            "required": ["date_start", "time_start", "venue", "price", "title"]
        }
    }
]

system_prompt = f"""
Ты — ассистент для извлечения данных из анонсов мероприятий. 
Используй функцию extract_event_data.
Текущая дата: {current_date}.
Правила:
- "в субботу в 19" → date_start = ближайшая суббота, time_start = 19:00
- "2500₽ (для своих 2000)" → price = 2500, price_discount = 2000
- Если поле неясно, верни null
"""

print("=== Результаты Function Calling ===")
for ex in examples:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": ex["text"]}
        ],
        functions=functions,
        function_call={"name": "extract_event_data"},
        temperature=0.1
    )
    print(f"\nПример {ex['id']}:")
    print(response.choices[0].message.function_call.arguments)
    print("-" * 50)

### Анализ результатов Function Calling

**Пример 1 (Идеальный):**
```json
{"date_start":"2026-09-19","time_start":"19:00","venue":"Пушкина 12","price":2500,"price_discount":2000,"title":"Разбираем тревогу у подростков"}

**Пример 2 (Честные null-поля):**
```json
{"date_start":"2026-09-23","time_start":null,"venue":null,"price":500,"price_discount":null,"title":"Встреча про выгорание"}

*✅ Модель не стала галлюцинировать адрес и время, а вернула null. Это правильное поведение.*
**Пример 3 (Конфликт даты):**
```json
{"date_start":"2025-11-25","time_start":"18:30","venue":"ул. Ленина 45, 3 этаж","price":3000,"price_discount":2500,"title":"Как строить здоровые границы в семье"}
*⚠️ Проблема: Модель вернула 2025 год, так как 25.11.2026 — это среда, а не вторник. Также в venue потерялось название "коворкинг «Среда»".*


## Подход C: Two-stage pipeline (Результаты генерации)

**Пример 1 → Email:**
> Прехедер: Встречаемся 19 сентября в 19:00 — разберём тревогу у подростков.
>
> Дорогие друзья!
> Приглашаю вас на встречу «Разбираем тревогу у подростков»...
>  19 сентября, 19:00
> 📍 Пушкина 12
> 💳 Стоимость — 2500 ₽

**Пример 2 → Email (Умная обработка null):**
> Здравствуйте!
> Приглашаю вас на встречу «Встреча про выгорание»...
> 📅 23 сентября
> 💰 Стоимость — 500 ₽
> **Время и место встречи уточняются.** Следите за обновлениями...

*✅ Модель сама дописала фразу про уточнение, увидев null в JSON. Это отлично решает боль рассинхронизации*

In [ ]:
# Замер реальной задержки Two-stage pipeline
import time

print("=== Замер реальной задержки ===\n")

# Stage 1: Extraction
start_time = time.time()

response_extraction = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": examples[0]["text"]}
    ],
    functions=functions,
    function_call={"name": "extract_event_data"},
    temperature=0.1
)

extraction_time = time.time() - start_time
print(f"Stage 1 (Extraction): {extraction_time:.2f} сек")

# Stage 2: Generation для ВК
start_time = time.time()

vk_prompt = f"""
Адаптируй анонс для ВКонтакте.
Правила:
- Заголовок до 100 символов
- Основной текст до 160 символов в сниппете
- Эмодзи допустимы (не больше 3)
- Тон: дружелюбный, приглашающий

Данные:
{response_extraction.choices[0].message.function_call.arguments}

Напиши пост для ВК.
"""

response_vk = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": vk_prompt}],
    temperature=0.7,
    max_tokens=300
)

generation_time = time.time() - start_time
print(f"Stage 2 (Generation ВК): {generation_time:.2f} сек")

# Итого
total_time = extraction_time + generation_time
print(f"\n⏱️ Итого Two-stage pipeline: {total_time:.2f} сек")
print(f"🎯 Лимит из PRD: 15 минут (900 сек)")
print(f"✅ Укладываемся: {'Да' if total_time < 900 else 'Нет'}")
print(f"📊 Запас: {900 / total_time:.0f} раз")

# Реальные токены
usage_extraction = response_extraction.usage
usage_vk = response_vk.usage
total_input = usage_extraction.prompt_tokens + usage_vk.prompt_tokens
total_output = usage_extraction.completion_tokens + usage_vk.completion_tokens

print(f"\n Реальные токены:")
print(f"  Input: {total_input} токенов")
print(f"  Output: {total_output} токенов")
print(f"  Всего: {total_input + total_output} токенов")

# Реальная стоимость
cost_rub = (total_input * 0.15 + total_output * 0.60) / 1_000_000 * 85
print(f"  Стоимость: {cost_rub:.4f} ₽ (лимит 10 ₽)")
print(f"  Запас: {10 / cost_rub:.0f} раз")